In [ ]:
##  一个售前和售后的 langchain  LLMRouterChain 模版

from langchain.chains.router import MultiPromptChain
from langchain_community.llms import Tongyi
from langchain.chains import ConversationChain
from langchain.chains.llm import LLMChain
from langchain.prompts import PromptTemplate
from langchain.chains.router.llm_router import (
    LLMRouterChain,
    RouterOutputParser
)
from langchain.chains.router.multi_prompt_prompt import (
    MULTI_PROMPT_ROUTER_TEMPLATE
)

# 售前咨询模板
presales_prompt_tpl = PromptTemplate.from_template(
    '你是一位专业的售前顾问，擅长产品介绍、方案推荐和商务咨询。'
    '你需要热情、专业地回答客户的产品咨询、价格询问、功能介绍等售前问题。'
    '请使用中文帮我解答下列售前咨询问题：\n{input}'
)

# 售后服务模板
aftersales_prompt_tpl = PromptTemplate.from_template(
    '你是一位耐心的售后服务专员，擅长解决客户的使用问题、技术支持和投诉处理。'
    '你需要耐心、细致地帮助客户解决产品使用中遇到的问题，提供技术支持和服务指导。'
    '请使用中文帮我解答下列售后服务问题：\n{input}'
)

# 创建模板信息列表
prompt_infos = [
    {
        'name': 'presales',
        'description': '用于处理售前咨询，包括产品介绍、价格询问、功能说明、方案推荐等',
        'prompt_template': presales_prompt_tpl,
    },
    {
        'name': 'aftersales',
        'description': '用于处理售后服务，包括使用问题、技术支持、故障排除、投诉处理等',
        'prompt_template': aftersales_prompt_tpl,
    },
]

llm = Tongyi(
    temperature=0.1,
)

# 生成键为模板名称、值为Chain的字典
destination_chains = {}
for p_info in prompt_infos:
    name = p_info['name']
    prompt = p_info['prompt_template']
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain

# 将模板名称和模板描述通过MULTI_PROMPT_ROUTER_TEMPLATE生成模板
destinations = [f'{p["name"]}: {p["description"]}'
                for p in prompt_infos]
destinations_str = "\n".join(destinations)
print(destinations_str)
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
print('=== 模板信息=== \n')
print(router_template)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=['input'],
    output_parser=RouterOutputParser(),
)
# 得到路由链：输入用户问题，输出一个目的地名。
# LLMRouterChain = “用 LLM 做路由决策”的那一小段链
router_chain = LLMRouterChain.from_llm(llm, router_prompt)

# 这里创建了一个default_chain
# 为了防止提的问题类型并没有包含在prompt_infos中
# 什么时候用 ConversationChain
# 需要多轮对话：当前回答要依赖历史几轮的 user/assistant 内容。
# 它负责把「对话历史」和「当前输入」一起塞进 prompt（或 buffer），不负责「选哪条链」。
default_chain = ConversationChain(llm=llm, output_key='text')
# 先跑router_chain得到目的地，再根据目的地选一条destination_chains里的链来跑，得到最终回答；若路由不到任何一条，就走default_chain（例如通用对话）
# 实现“多 prompt/多链 + 智能选择一条”的架构
# verbose=True：开启链的“详细日志/调试输出”。
# 在你这个 MultiPromptChain 场景下，它会把执行过程打印出来，比如：
# 路由器选中了哪个目的地链（例如 presales / aftersales / None）
# 进入/退出链的提示（类似 > Entering new MultiPromptChain chain...、> Finished chain.）
# 有时还会打印每一步的输入/中间结果（取决于具体链/版本）
chain = MultiPromptChain(
    router_chain=router_chain,
    destination_chains=destination_chains,
    default_chain=default_chain,
    verbose=True,
)

# 测试售前咨询问题
print("=== 售前咨询测试 ===")
print(chain.run("你们的产品有什么功能？价格是多少？"))

print("\n=== 售后服务测试 ===")
print(chain.run("我的产品出现故障了，无法正常启动，该怎么办？"))

print("\n=== 其他问题测试 ===")
print(chain.run("今天杭州天气怎么样？"))


presales: 用于处理售前咨询，包括产品介绍、价格询问、功能说明、方案推荐等
aftersales: 用于处理售后服务，包括使用问题、技术支持、故障排除、投诉处理等
=== 模板信息=== 

Given a raw text input to a language model select the model prompt best suited for the input. You will be given the names of the available prompts and a description of what the prompt is best suited for. You may also revise the original input if you think that revising it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}
```

REMEMBER: "destination" MUST be one of the candidate prompt names specified below OR it can be "DEFAULT" if the input is not well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
p

In [ ]:
# 整合链的语法

from langchain.chains.router import MultiPromptChain
from langchain_community.llms import Tongyi
from langchain.chains import ConversationChain
from langchain.chains.llm import LLMChain
from langchain.prompts import PromptTemplate
from langchain.chains.router.llm_router import (
    LLMRouterChain,
    RouterOutputParser
)
from langchain.chains.router.multi_prompt_prompt import (
    MULTI_PROMPT_ROUTER_TEMPLATE
)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# 初始化LLM
llm = Tongyi(temperature=0.1)

# 售前咨询链 - 使用新式语法
presales_prompt = PromptTemplate.from_template(
    '你是一位专业的售前顾问，擅长产品介绍、方案推荐和商务咨询。'
    '你需要热情、专业地回答客户的产品咨询、价格询问、功能介绍等售前问题。'
    '请使用中文帮我解答下列售前咨询问题：\n{input}'
)
presales_chain = presales_prompt | llm | StrOutputParser()

# 售后服务链 - 使用新式语法
aftersales_prompt = PromptTemplate.from_template(
    '你是一位耐心的售后服务专员，擅长解决客户的使用问题、技术支持和投诉处理。'
    '你需要耐心、细致地帮助客户解决产品使用中遇到的问题，提供技术支持和服务指导。'
    '请使用中文帮我解答下列售后服务问题：\n{input}'
)
aftersales_chain = aftersales_prompt | llm | StrOutputParser()

# 意图识别链 - 使用新式语法
intent_prompt = PromptTemplate.from_template(
    """请分析以下用户问题的意图，判断是售前咨询还是售后服务：

售前咨询：产品介绍、功能说明、价格询问、方案推荐、购买咨询等
售后服务：使用问题、技术支持、故障排除、投诉处理、维修服务等

用户问题：{input}

请只回答"售前"或"售后"，不要添加其他内容。"""
)
intent_chain = intent_prompt | llm | StrOutputParser()

# 创建路由函数
def route_question(input_dict):
    question = input_dict["input"]
    intent = intent_chain.invoke({"input": question})

    print(f"识别意图: {intent.strip()}")

    if "售前" in intent:
        return presales_chain.invoke({"input": question})
    elif "售后" in intent:
        return aftersales_chain.invoke({"input": question})
    else:
        # 默认处理
        default_prompt = PromptTemplate.from_template(
            "我是一个智能助手，很高兴为您服务。请问有什么可以帮助您的吗？\n问题：{input}"
        )
        default_chain = default_prompt | llm | StrOutputParser()
        return default_chain.invoke({"input": question})

# 创建完整的路由链
router_chain = RunnablePassthrough() | RunnableLambda(route_question)

# 方法二：更简洁的条件路由实现
from langchain_core.runnables import RunnableBranch

# 创建条件判断函数
def is_presales(input_dict):
    intent = intent_chain.invoke(input_dict)
    return "售前" in intent

def is_aftersales(input_dict):
    intent = intent_chain.invoke(input_dict)
    return "售后" in intent

# 使用 RunnableBranch 创建条件路由
# 是 LangChain LCEL 里的“条件分支路由器”（if/elif/else 版的 runnable）。
# 按顺序匹配：从上到下依次判断条件，命中第一个就执行对应分支，不会再看后面的条件。
# 条件是函数：cond(x) -> bool（可以是普通函数或 lambda），一般不做 LLM 调用（当然你也可以写，但会慢）。
# 分支是 runnable：每个分支可以是 prompt | llm | parser 这样的链，也可以是单个 runnable。
# 默认分支：最后可以放一个 runnable 作为兜底（else）。如果不放，且没有任何条件命中，通常会报错或行为不符合预期，所以生产里一般都给默认。
branch_chain = RunnableBranch(
    (is_presales, presales_chain),
    (is_aftersales, aftersales_chain),
    # 默认链
    PromptTemplate.from_template("我是智能助手，请问有什么可以帮助您的？\n{input}") | llm | StrOutputParser()
)

# 测试代码
if __name__ == "__main__":
    print("=== 方法一：自定义路由函数 ===")

    # 测试售前问题
    print("\n--- 售前咨询测试 ---")
    result1 = router_chain.invoke({"input": "你们的产品有什么功能？价格是多少？"})
    print(f"回答: {result1}")

    # 测试售后问题
    print("\n--- 售后服务测试 ---")
    result2 = router_chain.invoke({"input": "我的产品出现故障了，无法正常启动，该怎么办？"})
    print(f"回答: {result2}")

    print("\n=== 方法二：RunnableBranch 条件路由 ===")

    # 测试售前问题
    print("\n--- 售前咨询测试 ---")
    result3 = branch_chain.invoke({"input": "我想了解一下你们的服务套餐和收费标准"})
    print(f"回答: {result3}")

    # 测试售后问题
    print("\n--- 售后服务测试 ---")
    result4 = branch_chain.invoke({"input": "产品使用过程中遇到了错误提示，需要技术支持"})
    print(f"回答: {result4}")


=== 方法一：自定义路由函数 ===

--- 售前咨询测试 ---
识别意图: 售前
回答: 您好！非常感谢您的关注和咨询，很高兴为您介绍我们的产品 😊

我们提供的是**企业级智能协同办公平台「智联WorkPro」**（可根据您的行业需求定制为教育版、政务版、制造版等），它不是单一工具，而是一套“开箱即用、持续进化”的数字化工作底座。核心功能围绕**高效协作、智能提效、安全可控、开放集成**四大维度设计，具体包括：

✅ **核心功能亮点：**  
🔹 **智能文档协同**：支持多人实时在线编辑、版本追溯、AI辅助写作（自动生成会议纪要/周报/邮件）、敏感词自动识别与水印管控；  
🔹 **一体化项目管理**：甘特图+看板+工时统计，可关联OKR/KPI，支持任务自动拆解与风险预警；  
🔹 **智能会议中心**：一键预约会议室、AI语音转文字（支持中英日粤多语种）、会后自动生成待办与摘要，并同步至相关任务；  
🔹 **统一通讯与知识库**：集成IM即时消息、音视频会议、部门/项目级知识沉淀（支持全文检索、权限分级、知识图谱推荐）；  
🔹 **低代码流程引擎**：无需开发即可快速搭建审批流（如报销、采购、入职）、自动化报表及业务表单；  
🔹 **企业级安全与合规**：等保三级认证、数据本地化部署选项（私有云/混合云）、操作留痕审计、DLP数据防泄漏策略。

💡 **关于价格——我们坚持“价值匹配、透明灵活”原则：**  
我们不采用一刀切的标价模式，而是基于您的**实际组织规模、使用场景、部署方式及定制需求**提供个性化方案。参考如下主流配置（供您快速评估）：

| 版本类型       | 适用客户               | 起步价（年付） | 包含内容                                  |
|----------------|------------------------|----------------|-------------------------------------------|
| **标准云版**   | 50人以内成长型企业     | ¥19,800/年     | 全功能模块 + 基础AI能力 + 云端SAAS部署    |
| **专业私有版** | 中大型企业/对数据敏感单位 | ¥

## EmbeddingRouterChain

不仅可以使用 LLMRouteChain 来智能选择合适的处理链，还可以采用 EmbeddingRouterChain，该组件通过计算各 Chain 描述与用户问题之间的语义相关性，实现更精准的路由决策。


In [5]:
!pip install chromadb

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached importlib_metadata-8.7.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached zipp-3.23.0-py3-none-any.whl.metadata (3.6 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 35.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 17.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 27.2 MB/s eta 0:00:00a 0:00:01
Using cached importlib_metadata-8.7.1-py3-none-any.whl (27 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.8/751.8 kB 16.3 MB/s eta 0:00:00
Using cached zipp-3.23.0-py3-none-any.whl (10 kB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38/38 [chromadb]/38 [chromadb]a]antic-conventions]


### 使用官方 EmbeddingRouterChain

下面用 LangChain 官方的 `EmbeddingRouterChain` 作为 `MultiPromptChain` 的 `router_chain`，实现与上一格相同的「按语义选链」效果。

- **实现方式**：先手动用 Chroma 建向量库（与上一格相同逻辑），再 `EmbeddingRouterChain(vectorstore=..., routing_keys=["input"])`。这样既用到官方路由器，又避免 `from_names_and_descriptions` 在 Chroma 上可能出现的**维度冲突**（例如旧 collection 为 1536 维、当前 DashScope 为 1024 维会报 `InvalidArgumentError`）。使用独立 `collection_name` 可避免与其它单元格共用集合。
- **routing_keys=["input"]**：MultiPromptChain 传给 router 的输入 key 是 `"input"`，默认是 `["query"]`，故需显式指定。

In [8]:
# 使用官方 EmbeddingRouterChain + MultiPromptChain（与上一格效果一致）
# 说明：避免 from_names_and_descriptions，因 Chroma 可能复用旧 collection 导致维度冲突（如 1536 vs 1024）。
# 改为手动构建 vectorstore 再传入 EmbeddingRouterChain，并指定独立 collection_name 与 routing_keys=["input"]。
from langchain.chains.router import MultiPromptChain
from langchain.chains.router.embedding_router import EmbeddingRouterChain
from langchain.chains.llm import LLMChain
from langchain.chains import ConversationChain
from langchain.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_community.llms import Tongyi

# 1. 任务名称与描述（与上一格相同，用于路由匹配）
names_and_descriptions = [
    ("physics", ["用于解答物理相关问题，例如力学、电磁学等"]),
    ("math", ["用于解答数学相关问题，例如代数、几何、微积分等"]),
]

# 2. Embedding 模型
embeddings = DashScopeEmbeddings(model="text-embedding-v3")

# 3. 手动构建向量库（独立 collection_name 避免与其它单元格冲突导致维度错误），再创建 EmbeddingRouterChain
documents = []
for name, desc_list in names_and_descriptions:
    for desc in desc_list:
        documents.append(Document(page_content=desc, metadata={"name": name}))
texts = [doc.page_content for doc in documents]
metadatas = [doc.metadata for doc in documents]
vectorstore = Chroma(
    embedding_function=embeddings,
    collection_name="embedding_router_chain",  # 独立集合，避免 1536 vs 1024 维度冲突
)
vectorstore.add_texts(texts=texts, metadatas=metadatas)
router_chain = EmbeddingRouterChain(
    vectorstore=vectorstore,
    routing_keys=["input"],  # MultiPromptChain 传入的 key 是 "input"
)

# 4. 目标链的 prompt 与 LLM（与上一格一致，这里用 LLMChain 以兼容 MultiPromptChain）
llm = Tongyi(model_name="qwen-plus", temperature=0.1)
physics_prompt = PromptTemplate(
    template="你是一个物理专家，请回答以下问题：\n{input}",
    input_variables=["input"],
)
math_prompt = PromptTemplate(
    template="你是一个数学专家，请回答以下问题：\n{input}",
    input_variables=["input"],
)
destination_chains = {
    "physics": LLMChain(llm=llm, prompt=physics_prompt),
    "math": LLMChain(llm=llm, prompt=math_prompt),
}
default_chain = ConversationChain(llm=llm, output_key="text")

# 5. 组装 MultiPromptChain（与第一格 LLMRouterChain 用法一致，仅 router_chain 换成 EmbeddingRouterChain）
chain = MultiPromptChain(
    router_chain=router_chain,
    destination_chains=destination_chains,
    default_chain=default_chain,
    verbose=True,
)

# 6. 测试
print("=== EmbeddingRouterChain 测试 ===")
print(chain.run("牛顿第一定律是什么？"))
print("\n--- 数学问题 ---")
print(chain.run("什么是微积分基本定理？"))

=== EmbeddingRouterChain 测试 ===


> Entering new MultiPromptChain chain...
physics: {'input': '牛顿第一定律是什么？'}
> Finished chain.
牛顿第一定律，又称**惯性定律**，其标准表述为：

> **任何物体在不受外力作用（或所受合外力为零）时，总保持静止状态或匀速直线运动状态。**

### 关键要点解析：
1. **揭示惯性本质**：  
   定律指出物体具有**惯性**——即维持其原有运动状态（静止或匀速直线运动）的内在属性。质量是物体惯性大小的量度。

2. **定义惯性参考系**：  
   该定律**只在惯性参考系中成立**。换言之，牛顿第一定律不仅描述物体行为，更**定义了什么是惯性系**——即那些相对于“绝对空间”（经典力学中的理想背景）无加速度的参考系。现实中，地表参考系在精度要求不高时可近似视为惯性系；而加速运动的车厢、旋转地球（需考虑科里奥利效应）则不是严格惯性系。

3. **否定“力是维持运动的原因”这一错误观念**：  
   亚里士多德认为“运动需要力来维持”，而牛顿第一定律明确指出：**力不是维持速度的原因，而是改变速度（即产生加速度）的原因**。物体匀速直线运动无需力；只有当运动状态改变（加速、减速、转弯）时，才必然存在合外力。

4. **与牛顿第二定律的逻辑关系**：  
   第一定律是第二定律（\( \vec{F}_{\text{合}} = m\vec{a} \)）的特例：当 \( \vec{F}_{\text{合}} = 0 \) 时，必有 \( \vec{a} = 0 \)，故速度恒定（包括静止）。但第一定律具有独立意义——它确立了惯性系的存在和惯性概念，是整个牛顿力学体系的逻辑起点。

✅ **简记口诀**：  
*“无力不改态，静者恒静，动者恒直匀；有力才变速，惯性是本性。”*

如需进一步探讨其历史背景（伽利略斜面实验、笛卡尔贡献）、与相对论的联系，或常见误解辨析，欢迎继续提问！ 🌟

--- 数学问题 ---


> Entering new MultiPromptChain chain...
math: {'input': '什么是微积分基本定理？'}
> Finished chain.
微积分基

In [ ]:
from langchain_community.vectorstores import Chroma    # # pip install chroma
from langchain_community.embeddings import DashScopeEmbeddings # pip install dashscope
from langchain.chains import LLMRouterChain, MultiPromptChain
from langchain_core.language_models import BaseLLM
from langchain_core.prompts import PromptTemplate
from langchain_community.llms import Tongyi  # 或你使用的 LLM
import os

# 1. 定义任务名称与描述
names_and_descriptions = [
    ("physics", ["用于解答物理相关问题，例如力学、电磁学等"]),
    ("math", ["用于解答数学相关问题，例如代数、几何、微积分等"]),
]

# 2. 使用通义千问的 Embedding 模型
embeddings = DashScopeEmbeddings(model="text-embedding-v3")

# 3. 构建向量数据库（用于路由匹配）
descriptions = []
names = []
for name, desc_list in names_and_descriptions:
    for desc in desc_list:
        descriptions.append(desc)
        names.append(name)

# 创建 Chroma 向量库
vectorstore = Chroma(embedding_function=embeddings)
# 批量添加文档
vectorstore.add_texts(texts=descriptions, metadatas=[{"name": name} for name in names])

# 4. 自定义 Embedding 路由链（LangChain 没有直接 from_names_and_descriptions）
def get_relevant_chain_name(question: str) -> str:
    docs = vectorstore.similarity_search(question, k=1)
    return docs[0].metadata["name"]

# 5. 定义各个目标链的 prompt 和 LLM
llm = Tongyi(model_name="qwen-plus", temperature=0.1)  # 可替换为你用的 LLM

physics_prompt = PromptTemplate(
    template="你是一个物理专家，请回答以下问题：\n{input}",
    input_variables=["input"]
)
math_prompt = PromptTemplate(
    template="你是一个数学专家，请回答以下问题：\n{input}",
    input_variables=["input"]
)

destination_chains = {
    "physics": physics_prompt | llm,
    "math": math_prompt | llm,
}

default_chain = PromptTemplate.from_template("请回答以下问题：{input}") | llm

# 6. 定义运行逻辑（模拟 MultiPromptChain）
def run_router_chain(question: str):
    chain_name = get_relevant_chain_name(question)
    print(f"路由到: {chain_name}")
    if chain_name in destination_chains:
        return destination_chains[chain_name].invoke({"input": question})
    else:
        return default_chain.invoke({"input": question})

# 7. 测试
result = run_router_chain("牛顿第一定律是什么？")
print(result)

/var/folders/h6/fw8d9zmn0_b649z9ch8w4hy00000gn/T/ipykernel_36704/3447210662.py:27: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(embedding_function=embeddings)


路由到: physics
牛顿第一定律，又称**惯性定律**，其经典表述为：

> **任何物体在不受外力作用（或所受合外力为零）时，总保持静止状态或匀速直线运动状态。**

### 关键要点解析：
1. **揭示惯性本质**：  
   定律指出，物体具有保持其原有运动状态（静止或匀速直线运动）的内在属性，这种属性称为**惯性**。惯性的大小由物体的**质量**决定（质量越大，惯性越大）。

2. **定义惯性参考系**：  
   牛顿第一定律不仅描述物体行为，更**隐含地定义了惯性参考系**——即在该参考系中，若物体不受力（或合力为零），则加速度为零。只有在惯性系中，牛顿定律才严格成立。地面参考系在多数宏观低速情形下可近似视为惯性系；而加速运动的车厢、旋转地球等非惯性系中，需引入惯性力才能形式上维持定律适用。

3. **破除亚里士多德错误观念**：  
   该定律否定了“力是维持运动的原因”的古老误解，明确指出：**力不是维持速度的原因，而是改变运动状态（即产生加速度）的原因**。物体运动不需要持续受力——例如太空中的航天器关闭引擎后仍匀速飞行，正是此定律的体现。

4. **与牛顿第二定律的逻辑关系**：  
   第一定律可视为第二定律（\( \vec{F}_{\text{合}} = m\vec{a} \)）在 \( \vec{F}_{\text{合}} = 0 \) 时的特例（此时 \( \vec{a} = 0 \)），但其历史与哲学意义更为根本：它确立了力学的基本框架和时空观基础。

✅ 简记口诀：**“无力不改态，静者恒静，动者恒直匀。”**  

如需进一步探讨惯性系判据、非惯性系修正（如离心力/科里奥利力），或该定律在相对论中的适用范围（狭义相对论中仍成立，广义相对论中被推广为测地线运动），欢迎继续提问！ 🌟
